# ECG Anomaly Detection using LSTM Autoencoder with Temporal Attention Analysis

This cleaned portfolio notebook demonstrates the supplied normal-only LSTM Autoencoder, exact
backend-free reconstruction, reconstruction-error thresholding, anomaly evaluation, and post-hoc
temporal focus.

> **Healthcare disclaimer:** This is a synthetic educational demonstration. It is not a medical
> diagnostic tool and must not be used for clinical decisions.


## Attention qualification

The supplied pretrained model does not contain a trainable attention layer. The temporal-focus view
in this notebook is derived from pointwise reconstruction error. The optional retraining code in
`src/model_training.py` defines a true trainable temporal-attention architecture.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import SAMPLE_DATA_PATH, THRESHOLD, WEIGHTS_PATH
from src.data_preprocessing import frame_to_sequences, prepare_ecg_frame
from src.inference_pipeline import ECGInferenceService


## 1. Load the safe synthetic ECG sample

In [ ]:
ecg_frame = prepare_ecg_frame(pd.read_csv(SAMPLE_DATA_PATH))
signals = frame_to_sequences(ecg_frame)
ecg_frame[["signal_id", "label", "anomaly_type"]].head()


## 2. Load the pretrained cloud-safe inference service

In [ ]:
service = ECGInferenceService.from_artifacts(
    WEIGHTS_PATH,
    threshold=THRESHOLD,
)


## 3. Analyze one normal and one anomalous signal

In [ ]:
normal_index = int(ecg_frame.index[ecg_frame["label"] == 0][0])
anomaly_index = int(ecg_frame.index[ecg_frame["label"] == 1][0])

normal_result = service.analyze_signal(signals[normal_index])
anomaly_result = service.analyze_signal(signals[anomaly_index])

{
    "normal_error": normal_result.reconstruction_error,
    "normal_prediction": normal_result.predicted_status,
    "anomaly_error": anomaly_result.reconstruction_error,
    "anomaly_prediction": anomaly_result.predicted_status,
    "threshold": THRESHOLD,
}


In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(signals[anomaly_index].squeeze(), label="Original anomaly")
plt.plot(anomaly_result.reconstruction, label="Reconstruction")
plt.axhline(0.0, linestyle="--")
plt.xlabel("Timestep")
plt.ylabel("Amplitude")
plt.title("Original and Reconstructed Synthetic ECG-Like Signal")
plt.legend()
plt.tight_layout()
plt.show()


## 4. Inspect pointwise error and temporal focus

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(anomaly_result.pointwise_error)
plt.xlabel("Timestep")
plt.ylabel("Absolute reconstruction error")
plt.title("Pointwise Reconstruction Error")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(anomaly_result.temporal_focus)
plt.xlabel("Timestep")
plt.ylabel("Normalized temporal focus")
plt.title("Post-Hoc Temporal Focus")
plt.tight_layout()
plt.show()


## 5. Score the full packaged dataset

In [ ]:
scored = service.score_frame(ecg_frame)
scored.head()


In [ ]:
scored.groupby(["label", "predicted_label"]).size()


## Interpretation

A sequence is flagged when its mean absolute reconstruction error is at or above the training-normal
threshold. The packaged synthetic anomalies are deliberately simple, so performance is optimistic.
The result is not a medical diagnosis and should not be generalized to real ECG recordings.
